# Configure Files, Models, and Run Parameters

In [16]:
from pathlib import Path
import sys
import torch

INPUT_FILE = Path(r"C:\Users\zhang\tnl_project\facebook_comments3.csv")
OUTPUT_FILE = Path(r"C:\Users\zhang\tnl_project\translated.csv")

TEXT_COLUMN = "message"
TRANSLATION_COLUMN = "translated_message"

MODEL_NAME = "Helsinki-NLP/opus-mt-mul-en"

BATCH_SIZE = 32
SAVE_EVERY_BATCHES = 10
MAX_INPUT_TOKENS = 400
MAX_NEW_TOKENS = 256
NUM_BEAMS = 1
USE_FP16 = True

print("Input :", INPUT_FILE)
print("Output:", OUTPUT_FILE)

Input : C:\Users\zhang\tnl_project\facebook_comments3.csv
Output: C:\Users\zhang\tnl_project\translated.csv


# Read and Check the Dataset

In [17]:
import pandas as pd

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Cant find CSV：{INPUT_FILE}")

for encoding in ("utf-8", "utf-8-sig", "latin1", "cp1252"):
    try:
        source_df = pd.read_csv(INPUT_FILE, encoding=encoding)
        print("CSV encoding:", encoding)
        break
    except UnicodeDecodeError:
        continue
else:
    raise UnicodeError("Unable to recognize the character encoding of the CSV file.")

if TEXT_COLUMN not in source_df.columns:
    raise KeyError(f"Cant find '{TEXT_COLUMN}' column. Current columns：{source_df.columns.tolist()}")

print("Rows:", len(source_df))
print("Non-empty messages:", source_df[TEXT_COLUMN].fillna("").astype(str).str.strip().ne("").sum())
display(source_df[[TEXT_COLUMN]].head())

CSV encoding: utf-8
Rows: 6770
Non-empty messages: 6770


,message
0,Lelaki ini wajib disabitkan atas niat membunuh...
1,Lelaki ini wajib disabitkan atas niat membunuh...
2,Tak nak letak harapan tinggi dpd pihak berkuas...
3,Kes ustazah outo kena tersilap tekan minyak ni...
4,Muka x bersalah langsung …


# Load Translation Model

In [9]:
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = torch.device("cuda")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to(device).eval()

if USE_FP16:
    model = model.half()

print("Model:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("Data type:", next(model.parameters()).dtype)

C:\Users\zhang\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\zhang\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-mul-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Model: Helsinki-NLP/opus-mt-mul-en
Device: cuda:0
Data type: torch.float16


# Testing

In [11]:
sample_texts = (
    source_df[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .loc[lambda values: values.str.strip().ne("")]
    .head(3)
    .tolist()
)

sample_inputs = tokenizer(
    sample_texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=MAX_INPUT_TOKENS,
).to(device)

with torch.inference_mode():
    sample_outputs = model.generate(
        **sample_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
    )

sample_translations = tokenizer.batch_decode(
    sample_outputs,
    skip_special_tokens=True,
)

for original, translated in zip(sample_texts, sample_translations):
    print("ORIGINAL   :", original)
    print("TRANSLATED :", translated)
    print("-" * 80)

ORIGINAL   : Lelaki ini wajib disabitkan atas niat membunuh. Dia punca kemalangan  mengakibatkan 4 sekeluarga meninggal dunia di Johor. Dia adalah anak taukeh judi Simpang Renggam. Dan taukeh judi sekarang ada seorang anak berstatus pembunuh. Taukeh judi jangan cuba lari dari fakta, video dashcam ada sebagai bukti. 
Tak nak letak harapan tinggi dpd pihak berkuasa tapi harap polis tangani kes penuh dengan integriti. Jangan ada alasan2 tak masuk akal dibuat untuk pusing pusing bagi orang pening.
TRANSLATED : This man is supposed to be mentioned for his purpose of killing. He is a victim of the 4 families that died in Johor. He is a son of the great Simpang Rangem. And now a child is a murderer. The father does not try to run from the fact, the dashcam video is evidence. The high hopes of the authorities are not always high but the desire of the police to deal with the case is completely in the way of integrity. There is no reason why no one should be made to put pressure on the person.
-

# Translate Dataset

In [25]:
import time

def save_checkpoint(dataframe, output_path):
    """Safely save current translation progress."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")
    dataframe.to_csv(temporary_path, index=False, encoding="utf-8-sig")
    temporary_path.replace(output_path)


# Resume from a previous output when possible
if OUTPUT_FILE.exists():
    existing_df = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")

    if len(existing_df) == len(source_df) and TRANSLATION_COLUMN in existing_df.columns:
        df = source_df.copy()
        df[TRANSLATION_COLUMN] = existing_df[TRANSLATION_COLUMN]
        print("Loaded previous checkpoint:", OUTPUT_FILE)
    else:
        raise ValueError(
            "The number of rows or columns in the output file does not match that of the input file."
            "Please rename or delete the old output file first."
        )
else:
    df = source_df.copy()
    df[TRANSLATION_COLUMN] = ""

messages = df[TEXT_COLUMN].fillna("").astype(str)
translations = df[TRANSLATION_COLUMN].fillna("").astype(str)

# Empty source rows do not require model inference
empty_source = messages.str.strip().eq("")
df.loc[empty_source, TRANSLATION_COLUMN] = ""

pending_indices = df.index[
    messages.str.strip().ne("") & translations.str.strip().eq("")
].tolist()

# Similar lengths in the same batch reduce unnecessary padding
pending_indices.sort(key=lambda index: len(messages.at[index]))

total_pending = len(pending_indices)
print("Total rows:", len(df))
print("Already translated:", len(df) - total_pending - int(empty_source.sum()))
print("Waiting for translation:", total_pending)

started = time.perf_counter()

for batch_number, start in enumerate(range(0, total_pending, BATCH_SIZE), start=1):
    batch_indices = pending_indices[start:start + BATCH_SIZE]
    batch_texts = [messages.at[index] for index in batch_indices]

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(device)

    with torch.inference_mode():
        generated_tokens = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
        )

    decoded_batch = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    for row_index, translated_text in zip(batch_indices, decoded_batch):
        df.at[row_index, TRANSLATION_COLUMN] = translated_text

    completed = min(start + len(batch_indices), total_pending)
    elapsed = time.perf_counter() - started
    rows_per_second = completed / elapsed if elapsed else 0
    remaining_seconds = (total_pending - completed) / rows_per_second if rows_per_second else 0

    print(
        f"Batch {batch_number} | "
        f"{completed}/{total_pending} | "
        f"{rows_per_second:.2f} rows/s | "
        f"ETA {remaining_seconds / 60:.1f} min"
    )

    if batch_number % SAVE_EVERY_BATCHES == 0:
        save_checkpoint(df, OUTPUT_FILE)
        print("Checkpoint saved.")

save_checkpoint(df, OUTPUT_FILE)

elapsed = time.perf_counter() - started
print("\nTranslation completed.")
print("Output:", OUTPUT_FILE.resolve())
print("This run:", total_pending, "rows")
print("Time:", round(elapsed / 60, 2), "minutes")

Total rows: 6770
Already translated: 0
Waiting for translation: 6770
Batch 1 | 32/6770 | 62.41 rows/s | ETA 1.8 min
Batch 2 | 64/6770 | 84.81 rows/s | ETA 1.3 min
Batch 3 | 96/6770 | 99.14 rows/s | ETA 1.1 min
Batch 4 | 128/6770 | 101.76 rows/s | ETA 1.1 min
Batch 5 | 160/6770 | 109.59 rows/s | ETA 1.0 min
Batch 6 | 192/6770 | 114.66 rows/s | ETA 1.0 min
Batch 7 | 224/6770 | 114.10 rows/s | ETA 1.0 min
Batch 8 | 256/6770 | 113.93 rows/s | ETA 1.0 min
Batch 9 | 288/6770 | 113.45 rows/s | ETA 1.0 min
Batch 10 | 320/6770 | 111.08 rows/s | ETA 1.0 min
Checkpoint saved.
Batch 11 | 352/6770 | 109.32 rows/s | ETA 1.0 min
Batch 12 | 384/6770 | 103.88 rows/s | ETA 1.0 min
Batch 13 | 416/6770 | 103.06 rows/s | ETA 1.0 min
Batch 14 | 448/6770 | 101.65 rows/s | ETA 1.0 min
Batch 15 | 480/6770 | 102.05 rows/s | ETA 1.0 min
Batch 16 | 512/6770 | 101.99 rows/s | ETA 1.0 min
Batch 17 | 544/6770 | 100.45 rows/s | ETA 1.0 min
Batch 18 | 576/6770 | 99.60 rows/s | ETA 1.0 min
Batch 19 | 608/6770 | 98.75 r